In [9]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

import optuna

In [10]:
train_df = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\train_features.csv')
val_df = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\validation_features.csv')
test_df = pd.read_csv(r'D:\Creditcard_fraud_detection\data\processed\test_features.csv')

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (156011, 52)
Validation: (33431, 52)
Test: (33431, 52)


In [11]:
TARGET = "is_fraud"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("Validation features:", X_val.shape)
print("Validation target:", y_val.shape)

print("Test features:", X_test.shape)
print("Test target:", y_test.shape)

Training features: (156011, 51)
Training target: (156011,)
Validation features: (33431, 51)
Validation target: (33431,)
Test features: (33431, 51)
Test target: (33431,)


In [12]:
forbidden_columns = [
    "cc_num",
    "trans_num",
    "Unnamed: 0",
    "first",
    "last",
    "street",
    "trans_date_trans_time",
    "dob"
]

X_train = X_train.drop(columns=forbidden_columns,errors="ignore")
X_val = X_val.drop(columns=forbidden_columns,errors="ignore")
X_test = X_test.drop(columns=forbidden_columns,errors="ignore")
print("Remaining features:", X_train.shape[1])

Remaining features: 47


In [13]:
categorical_candidates = [
    "merchant",
    "category",
    "gender",
    "city",
    "state",
    "job",
    "zip",
    "amount_bucket"
]

categorical_features = [
    col for col in categorical_candidates
    if col in X_train.columns
]
numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:",len(categorical_features))
print("\nNumber of numerical features:",len(numeric_features))

Categorical features:
['merchant', 'category', 'gender', 'city', 'state', 'job', 'zip', 'amount_bucket']

Number of categorical features: 8

Number of numerical features: 39


In [14]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore',sparse_output=True))
])

preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline,numeric_features),
    ("categorical",categorical_pipeline, categorical_features)
])


In [15]:
X_train_transformed = preprocessor.fit_transform(X_train)

X_val_transformed = preprocessor.transform(X_val)

print("Train transformed shape:", X_train_transformed.shape)
print("Validation transformed shape:", X_val_transformed.shape)

Train transformed shape: (156011, 3120)
Validation transformed shape: (33431, 3120)


In [16]:
preprocessor.fit_transform(X_train)
preprocessor.transform(X_val)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1568089 stored elements and shape (33431, 3120)>

In [17]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Legitimate:", negative_count)
print("Fraud:", positive_count)
print("Scale Pos Weight:", scale_pos_weight)

Legitimate: 150890
Fraud: 5121
Scale Pos Weight: 29.464948252294473


In [18]:
def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),
        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10, log=True
        ),

        "scale_pos_weight": scale_pos_weight,

        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "tree_method": "hist",
        "n_jobs": 2,
        "random_state": 42
    }

    model = XGBClassifier(**params)

    model.fit(X_train_transformed,y_train,eval_set=[(X_val_transformed, y_val)],verbose=False)

    y_val_proba = model.predict_proba(X_val_transformed)[:, 1]

    return average_precision_score(y_val,y_val_proba)

In [ ]:
study_xgb = optuna.create_study(direction="maximize",study_name="XGBoost_Fraud")

study_xgb.optimize(objective_xgb,n_trials=25)

[I 2026-09-14 00:10:25,380] A new study created in memory with name: XGBoost_Fraud
[I 2026-09-14 00:15:48,774] Trial 0 finished with value: 0.9855544628472526 and parameters: {'n_estimators': 152, 'max_depth': 6, 'learning_rate': 0.028279900871265102, 'min_child_weight': 3, 'subsample': 0.7067062139760091, 'colsample_bytree': 0.9084327983942163, 'gamma': 0.401970033319512, 'reg_alpha': 0.0028786674512336916, 'reg_lambda': 0.002174758693939186}. Best is trial 0 with value: 0.9855544628472526.
[I 2026-09-14 00:22:17,999] Trial 1 finished with value: 0.9908575562180144 and parameters: {'n_estimators': 187, 'max_depth': 5, 'learning_rate': 0.06841507228892618, 'min_child_weight': 7, 'subsample': 0.7155179961885488, 'colsample_bytree': 0.8807928540229248, 'gamma': 0.955813564125314, 'reg_alpha': 0.002896260040066483, 'reg_lambda': 5.899697807465605}. Best is trial 1 with value: 0.9908575562180144.
[I 2026-09-14 00:28:07,760] Trial 2 finished with value: 0.9617119434140798 and parameters: {'

## Hyperparameter Optimization — Computational Constraint

Hyperparameter optimization was performed using Optuna to identify suitable
model configurations.

However, due to the computational limitations of the local development
environment (8 GB RAM and an older Intel i5 processor), extensive
hyperparameter optimization was not computationally feasible.

The Optuna search was therefore limited to a small number of trials to
prevent excessive memory consumption and prolonged training times.

This limitation was considered when designing the experiment. The objective
was to identify a strong model configuration while maintaining reasonable
computational requirements.

The best-performing configurations from the available trials were retained
for subsequent model evaluation.

> **Computational limitation:** Extensive hyperparameter optimization could
> not be completed on the local hardware. Therefore, the results should be
> interpreted as a limited hyperparameter search rather than an exhaustive
> optimization.

In [ ]:
#